# BAH Dataset — Video Clip Training (3D-CNN / TimeSformer)

Trains a video-clip classifier on the BAH dataset.

**Modalities (choose one via `MODALITY`):**
- `'frames'`           — clips from `data/Frames/` (raw frames)
- `'segmented_frames'` — clips from `data/SegmentedFrames/` (YOLO body crops; `.skip` frames excluded)

**Each sample:** `(clip_tensor, label)` where `clip_tensor` has shape `(clip_len, C, H, W)`.

**Pipeline:**
1. Build clips of `CLIP_LEN` consecutive frames per video using `get_video_clip_splits()` or `get_segmented_video_clip_splits()`
2. Encode each clip with a lightweight 3D-CNN (R3D-18 from torchvision) or swap in any model that accepts `(B, C, T, H, W)`
3. Evaluate on val split after every epoch; save best weights by macro-F1
4. Final inference on test split

In [ ]:
import os, sys, random, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# ── Paths — run from repo root or adjust sys.path ────────────────────────────
REPO_ROOT = Path('..').resolve()          # video/src/ → repo root
sys.path.insert(0, str(REPO_ROOT / 'utils'))
from load_dataset import (
    get_video_clip_splits,
    get_segmented_video_clip_splits,
    CLASS_NAMES, NUM_CLASSES,
)

# ── Config ────────────────────────────────────────────────────────────────────
# Choose modality: 'frames' or 'segmented_frames'
MODALITY     = 'segmented_frames'

IMAGE_SIZE   = 112          # spatial resolution per frame
CLIP_LEN     = 16           # frames per clip  (15–30 recommended)
CLIP_STRIDE  = 0            # 0 = non-overlapping (stride = clip_len)
PAD_LAST     = True         # pad the last partial clip with zeros

IMG_MEAN     = [0.485, 0.456, 0.406]
IMG_STD      = [0.229, 0.224, 0.225]

EPOCHS       = 20
BATCH_SIZE   = 8            # each sample is (clip_len, C, H, W) — keep small
LR           = 1e-4
WD           = 1e-4
NUM_WORKERS  = 0
SEED         = 42
PATIENCE     = 5            # early-stopping patience (0 = disabled)

WEIGHTS_DIR  = Path('../weights')
WEIGHTS_NAME = f'r3d18_{MODALITY}_bah_best.pth'
LAST_NAME    = f'r3d18_{MODALITY}_bah_last.pth'

# ── Reproducibility ───────────────────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Device ────────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

print(f'Device    : {DEVICE}')
print(f'Modality  : {MODALITY}')
print(f'Clip len  : {CLIP_LEN} frames  |  Image size : {IMAGE_SIZE}px')
print(f'Epochs    : {EPOCHS}  |  Batch : {BATCH_SIZE}  |  LR : {LR}')

In [ ]:
# ── Per-frame transforms ──────────────────────────────────────────────────────
# Applied to each frame individually inside the Dataset.__getitem__.
# The Dataset returns a numpy HWC uint8 image; ToPILImage converts it first.
train_frame_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMG_MEAN, std=IMG_STD),
])

eval_frame_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMG_MEAN, std=IMG_STD),
])

# ── Datasets ──────────────────────────────────────────────────────────────────
common_kw = dict(
    image_size   = IMAGE_SIZE,
    clip_len     = CLIP_LEN,
    clip_stride  = CLIP_STRIDE,
    pad_last     = PAD_LAST,
)

if MODALITY == 'segmented_frames':
    splits = get_segmented_video_clip_splits(
        train_transform=train_frame_transform,
        eval_transform=eval_frame_transform,
        **common_kw,
    )
else:
    splits = get_video_clip_splits(
        train_transform=train_frame_transform,
        eval_transform=eval_frame_transform,
        **common_kw,
    )

train_ds, val_ds, test_ds = splits['train'], splits['val'], splits['test']

# ── DataLoaders ───────────────────────────────────────────────────────────────
_pw = NUM_WORKERS > 0
_pm = DEVICE.type == 'cuda'

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=_pm,
    persistent_workers=_pw, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=_pm,
    persistent_workers=_pw,
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=_pm,
    persistent_workers=_pw,
)

print(f'Train batches : {len(train_loader):,}')
print(f'Val   batches : {len(val_loader):,}')
print(f'Test  batches : {len(test_loader):,}')

# ── Sanity check: batch shape ─────────────────────────────────────────────────
clips, labels = next(iter(train_loader))
print(f'Clip batch shape : {clips.shape}   # (B, T, C, H, W)')
print(f'Labels sample    : {labels[:8].tolist()}')

In [ ]:
# ── Visualise a sample clip ───────────────────────────────────────────────────
def denorm(t):
    mean = torch.tensor(IMG_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMG_STD).view(3, 1, 1)
    return (t * std + mean).clamp(0, 1)

clip_idx = 0
clip_frames = clips[clip_idx]          # (T, C, H, W)
n_show = min(8, CLIP_LEN)
fig, axes = plt.subplots(1, n_show, figsize=(n_show * 2, 2.5))
for i, ax in enumerate(axes):
    img_np = denorm(clip_frames[i]).permute(1, 2, 0).numpy()
    ax.imshow(img_np)
    ax.set_title(f'f{i}', fontsize=7)
    ax.axis('off')
plt.suptitle(
    f'Sample clip — label: {CLASS_NAMES[labels[clip_idx].item()]}  '
    f'(modality: {MODALITY})',
    fontweight='bold',
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Model: R3D-18 (torchvision) ───────────────────────────────────────────────
# R3D-18 expects input shape (B, C, T, H, W).
# Our DataLoader yields (B, T, C, H, W) — we permute inside the training loop.
from torchvision.models.video import r3d_18, R3D_18_Weights

model = r3d_18(weights=R3D_18_Weights.DEFAULT)
# Replace the final FC layer for binary classification
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, NUM_CLASSES)
model = model.to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'R3D-18 total params     : {total_params:,}')
print(f'R3D-18 trainable params : {trainable_params:,}')

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=LR * 1e-2,
)
print(f'Optimizer : AdamW  lr={LR}  wd={WD}')
print(f'Scheduler : CosineAnnealingLR  T_max={EPOCHS}')

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred):
    return {
        'acc'      : accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall'   : recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1'       : f1_score(y_true, y_pred, average='macro', zero_division=0),
    }


def run_epoch(loader, model, criterion, optimizer=None, desc=''):
    """One training or evaluation pass.

    Clips arrive as (B, T, C, H, W); R3D-18 needs (B, C, T, H, W).
    """
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    pbar = tqdm(loader, desc=desc, leave=False, unit='batch')
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for clips, labels in pbar:
            # (B, T, C, H, W) → (B, C, T, H, W)
            clips  = clips.permute(0, 2, 1, 3, 4).to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(clips)
            loss   = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * len(labels)
            all_preds.extend(logits.argmax(1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
            pbar.set_postfix(loss=f'{loss.item():.4f}')

    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, compute_metrics(all_labels, all_preds)


WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

history     = defaultdict(list)
best_val_f1 = -1.0
no_improve  = 0

header = (
    f"{'Epoch':>6} {'TrLoss':>8} {'TrAcc':>7} {'TrF1':>7} "
    f"{'VaLoss':>8} {'VaAcc':>7} {'VaPrec':>8} {'VaRec':>7} {'VaF1':>7} {'LR':>10}"
)
print(f'Starting training for {EPOCHS} epochs on {DEVICE}...')
print('─' * len(header))
print(header)
print('─' * len(header))

epoch_bar = tqdm(range(1, EPOCHS + 1), desc='Epochs', unit='epoch')
for epoch in epoch_bar:
    tr_loss, tr_m = run_epoch(
        train_loader, model, criterion, optimizer,
        desc=f'Ep {epoch}/{EPOCHS} train',
    )
    va_loss, va_m = run_epoch(
        val_loader, model, criterion,
        desc=f'Ep {epoch}/{EPOCHS} val  ',
    )
    epoch_bar.set_postfix(val_f1=f'{va_m["f1"]:.4f}', val_acc=f'{va_m["acc"]:.4f}')

    if scheduler is not None:
        scheduler.step()

    cur_lr = optimizer.param_groups[0]['lr']

    history['epoch'].append(epoch)
    history['tr_loss'].append(tr_loss)
    history['va_loss'].append(va_loss)
    history['lr'].append(cur_lr)
    for k, v in tr_m.items():
        history[f'tr_{k}'].append(v)
    for k, v in va_m.items():
        history[f'va_{k}'].append(v)

    print(
        f"{epoch:>6} {tr_loss:>8.4f} {tr_m['acc']:>7.4f} {tr_m['f1']:>7.4f} "
        f"{va_loss:>8.4f} {va_m['acc']:>7.4f} {va_m['precision']:>8.4f} "
        f"{va_m['recall']:>7.4f} {va_m['f1']:>7.4f} {cur_lr:>10.2e}"
    )

    if va_m['f1'] > best_val_f1:
        best_val_f1 = va_m['f1']
        torch.save(model.state_dict(), WEIGHTS_DIR / WEIGHTS_NAME)
        print(f'  New best val F1={best_val_f1:.4f} — saved {WEIGHTS_NAME}')
        no_improve = 0
    else:
        no_improve += 1

    if PATIENCE > 0 and no_improve >= PATIENCE:
        print(f'Early stopping after {epoch} epochs (patience={PATIENCE})')
        break

print('─' * len(header))
print(f'Training complete.  Best val F1 = {best_val_f1:.4f}')
torch.save(model.state_dict(), WEIGHTS_DIR / LAST_NAME)
print(f'Last-epoch weights saved -> {WEIGHTS_DIR / LAST_NAME}')

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
hist = pd.DataFrame(history)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

axes[0, 0].plot(hist['epoch'], hist['tr_loss'], label='Train')
axes[0, 0].plot(hist['epoch'], hist['va_loss'], label='Val')
axes[0, 0].set_title('Loss')
axes[0, 0].legend()
axes[0, 0].set_xlabel('Epoch')

axes[0, 1].plot(hist['epoch'], hist['tr_acc'], label='Train')
axes[0, 1].plot(hist['epoch'], hist['va_acc'], label='Val')
axes[0, 1].set_title('Accuracy')
axes[0, 1].legend()
axes[0, 1].set_xlabel('Epoch')

axes[1, 0].plot(hist['epoch'], hist['tr_f1'], label='Train')
axes[1, 0].plot(hist['epoch'], hist['va_f1'], label='Val')
axes[1, 0].set_title('F1 (macro)')
axes[1, 0].legend()
axes[1, 0].set_xlabel('Epoch')

axes[1, 1].plot(hist['epoch'], hist['va_precision'], label='Precision')
axes[1, 1].plot(hist['epoch'], hist['va_recall'],    label='Recall')
axes[1, 1].plot(hist['epoch'], hist['va_f1'],        label='F1')
axes[1, 1].set_title('Val Precision / Recall / F1')
axes[1, 1].legend()
axes[1, 1].set_xlabel('Epoch')

for ax in axes.flat:
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(alpha=0.3)

plt.suptitle(
    f'Video Clip (R3D-18, {MODALITY}) — Training history',
    fontsize=14, fontweight='bold',
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Test inference ────────────────────────────────────────────────────────────
best_ckpt = WEIGHTS_DIR / WEIGHTS_NAME
model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
print(f'Loaded best weights from {best_ckpt}')

model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for clips, labels in tqdm(test_loader, desc='Test inference'):
        clips  = clips.permute(0, 2, 1, 3, 4).to(DEVICE)   # (B,C,T,H,W)
        logits = model(clips)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()
        preds  = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())
        all_probs.extend(probs.tolist())

test_m = compute_metrics(all_labels, all_preds)

print()
print('=' * 50)
print('TEST SET METRICS')
print('=' * 50)
print(f"  Accuracy  : {test_m['acc']:.4f}")
print(f"  Precision : {test_m['precision']:.4f}  (macro)")
print(f"  Recall    : {test_m['recall']:.4f}  (macro)")
print(f"  F1        : {test_m['f1']:.4f}  (macro)")
print()
print('Per-class report:')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4))

# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES)
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Test Set', fontweight='bold')
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, f'{cm[i, j]:,}', ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=12)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()